# Seven RCTs of microcredit

This notebook reproduces the following two sections of the original paper using Python and the JAX library.
- 4.3 Seven RCTs of microcredit: Linear regression analysis
- 4.4 Seven RCTs of microcredit: Bayesian hierarchical tailored mixture model

In [ ]:
from pathlib import Path
import subprocess
import tempfile
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import pandas as pd1
from statistics import NormalDist
from IPython.display import display
z = NormalDist().inv_cdf(0.975)
DATA_FILE = Path("/Users/yumin/Downloads/aggregating-dist-TEs/data/microcredit_project_data.RData")

# Data Preparation
The data processing logic was adopted from the authors' [AMIP preprocessing script](https://github.com/rgiordan/AMIPPaper/blob/main/writing/applications/microcredit/preprocess_microcredit_data.R). 

In [1]:
r_export = r"""
# Load into a separate environment to avoid overwriting existing variables.
e <- new.env()
load(
  "/Users/yumin/Downloads/aggregating-dist-TEs/data/microcredit_project_data.RData",
  envir = e
)

# Inspect the available objects.
ls(e)

prefixes <- c("angelucci", "attanasio", "augsberg", "banerjee", "crepon", "karlan", "tarozzi")

print(e$study_country)
# Mexico, Mongolia, Bosnia, India, Morocco, Philippines, Ethiopia

prepare_outcome <- function(outcome) {
  K <- if (outcome == "profit") 7 else 5

  standardisers <- e[[
    paste0("the_", outcome, "_standardiser_USD_PPP_per_fortnight")
  ]]

  pieces <- lapply(seq_len(K), function(k) {
    y <- e[[paste0(prefixes[k], "_", outcome)]]
    treatment <- e[[paste0(prefixes[k], "_treatment")]]

    stopifnot(length(y) == length(treatment))

    d <- data.frame(
      site = k,
      country = e$study_country[k],
      source_row = seq_along(y),
      treatment = treatment,
      y = y * standardisers[k]
    )

    d[complete.cases(d), ]
  })

  do.call(rbind, pieces)
}

profit_data <- prepare_outcome("profit")
temptation_data <- prepare_outcome("temptation")

dir.create("data/microcredit", recursive = TRUE, showWarnings = FALSE)

write.csv(
  profit_data, "data/microcredit/profit.csv", row.names = FALSE
)
write.csv(
  temptation_data, "data/microcredit/temptation.csv", row.names = FALSE
)"""

In [5]:
with tempfile.TemporaryDirectory() as tmp:
    script = Path(tmp) / "export.R"
    script.write_text(r_export)
    subprocess.run(["Rscript", str(script), str(DATA_FILE), tmp], check=True)

  [1] "additional_loans_data_attanasio_2015"                   
  [2] "angelucci_APR"                                          
  [3] "angelucci_assets"                                       
  [4] "angelucci_averageloansize"                              
  [5] "angelucci_bank_loan_amount"                             
  [6] "angelucci_bank_loan_indicator"                          
  [7] "angelucci_businessincome"                               
  [8] "angelucci_businessincome_non_imputed"                   
  [9] "angelucci_cluster_id"                                   
 [10] "angelucci_collateralised"                               
 [11] "angelucci_consumption"                                  
 [12] "angelucci_consumption_source_dataframe"                 
 [13] "angelucci_currentmarket"                                
 [14] "angelucci_district"                                     
 [15] "angelucci_eduspend"                                     
 [16] "angelucci_existingbusiness"      

Import the data into python and check the samples.

In [6]:
data_dir = Path("data/microcredit")

datasets = {
    "profit": pd.read_csv(data_dir / "profit.csv"),
    "temptation": pd.read_csv(data_dir / "temptation.csv"),
}

for outcome, df in datasets.items():
    assert set(df["treatment"].unique()) == {0, 1}
    assert df[["y", "treatment"]].notna().all().all()

    print(outcome)
    print(df.groupby("country").size())

profit
country
Bosnia          1195
Ethiopia        3113
India           6863
Mexico         16560
Mongolia         961
Morocco         5498
Philippines     1113
dtype: int64
temptation
country
Bosnia        996
India        6827
Mexico      16435
Mongolia      961
Morocco      5487
dtype: int64


# Model Definition

For each country, we fit the model 

$Y_i=\beta_0+\beta_1T_i+\epsilon_i$,

where $\beta_1$ represent the treatment-group mean minus the control-group mean.

In [ ]:
def fit_ols(w, X, y):
    """Return the treatment coefficient and its standard error"""
    A = X.T @ (w[:, None] * X)

    beta = jnp.linalg.solve(
        A,
        X.T @ (w * y),
    )

    residual = y - X @ beta

    # Use Zero-weight refit convention from the author
    sigma2 = jnp.sum(w * residual**2) / (y.shape[0]-X.shape[1])

    covariance = sigma2 * jnp.linalg.inv(A)
    treatment_se = jnp.sqrt(covariance[1,1])

    return jnp.array([beta[1], treatment_se])

# Model Fitting: Mexico profit
We carry on with a subset of data to ensure that the pipeline works as intended, before applying it on all of the profit and temptation datasets. 

In [8]:
d = datasets["profit"].loc[datasets["profit"]["country"] == "Mexico"].reset_index(drop=True)

y = jnp.asarray(d["y"].to_numpy())

X = jnp.column_stack([jnp.ones(len(d)), jnp.asarray(d["treatment"].to_numpy())])

w = jnp.ones(len(d))

# A function of deletion weights alone
fit = lambda weights: fit_ols(weights, X, y)

b, se = np.asarray(fit(w))

print(f"Estimate: {b:.3f}")
print(f"Standard error: {se:.3f}")

Estimate: -4.549
Standard error: 5.879


The model produces the estimate and SE that aligns with the paper! So we proceed. 

# Differentiation
The fitting function returns two quantities. We differentiate both with respect to all observation weights.

In [9]:
derivatives = np.asarray(jax.jacrev(fit)(w))

db = derivatives[0] # d beta_1 / d w_i
ds = derivatives[1] # d SE / d w_i

print(derivatives.shape)

(2, 16560)


# Conclusion Targets

In [10]:
z = NormalDist().inv_cdf(0.975)

def make_targets(b, se, db, ds):
    L = b - z * se
    U = b + z * se

    return {
        "Sign change": (b, db),
        "Significance change": ((L, db - z * ds) if abs(L) < abs(U) else (U, db + z * ds)),
        "Significant sign change": ((U, db + z * ds) if b >= 0 else (L, db - z * ds))
    }

# Rank Observations and Predicted Deletion Count

In [11]:
def select_subset(q, psi):
    gain = np.sign(q) * np.asarray(psi)

    order = np.argsort(-gain, kind="stable")
    order = order[gain[order] > 0]

    cumulative_gain = np.cumsum(gain[order])
    hits = np.flatnonzero(cumulative_gain >= abs(q))

    if len(hits) == 0:
        return None
    
    k = int(hits[0]) + 1
    return order[:k]

In [17]:
def analyse_country(d):
    d = d.reset_index(drop=True)

    y = jnp.asarray(d["y"].to_numpy())
    X = jnp.column_stack([
        jnp.ones(len(d)),
        jnp.asarray(d["treatment"].to_numpy())
    ])

    w = jnp.ones(len(d))
    fit = lambda weights: fit_ols(weights, X, y)

    b, se = np.asarray(fit(w))
    db, ds = np.asarray(jax.jacrev(fit)(w))

    records = []

    for target, (q, psi) in make_targets(b, se, db, ds).items():
        drop = select_subset(q, psi)

        if drop is None:
            raise RuntimeError(f"Unreachable linear target: {target}")
        
        refit_w = w.at[drop].set(0.0)
        br, sr = np.asarray(fit(refit_w))

        significant_before = abs(b) > z * se
        significant_after = abs(br) > z * sr

        achieved = {
            "Sign change": b * br < 0,
            "Significance change": significant_after != significant_before,
            "Significant sign change": (b * br < 0) and significant_after,
        }[target]

        records.append({
            "country": d["country"].iloc[0],
            "N": len(d),
            "target": target,
            "estimate": b,
            "se": se,
            "dropped": len(drop),
            "percent": 100 * len(drop) / len(d),
            "predicted_target": q - np.asarray(psi)[drop].sum(),
            "refit_estimate": br,
            "refit_se": sr,
            "achieved": bool(achieved),
            "dropped_source_rows": d["source_row"].iloc[drop].tolist(),
        })
    
    return records

In [18]:
analyse_country(d)

[{'country': 'Mexico',
  'N': 16560,
  'target': 'Sign change',
  'estimate': np.float64(-4.549115856025455),
  'se': np.float64(5.878890737385146),
  'dropped': 1,
  'percent': 0.006038647342995169,
  'predicted_target': np.float64(0.396932235797685),
  'refit_estimate': np.float64(0.39753095848153513),
  'refit_se': np.float64(3.193559792041455),
  'achieved': True,
  'dropped_source_rows': [9799]},
 {'country': 'Mexico',
  'N': 16560,
  'target': 'Significance change',
  'estimate': np.float64(-4.549115856025455),
  'se': np.float64(5.878890737385146),
  'dropped': 14,
  'percent': 0.08454106280193237,
  'predicted_target': np.float64(-0.0296527632300414),
  'refit_estimate': np.float64(-10.962140748814154),
  'refit_se': np.float64(5.56517882901641),
  'achieved': True,
  'dropped_source_rows': [18444,
   20933,
   10821,
   13499,
   15650,
   8640,
   8897,
   7698,
   21107,
   8667,
   13656,
   9867,
   18430,
   13131]},
 {'country': 'Mexico',
  'N': 16560,
  'target': 'Signi

In accordance with the results provided in Table 4. We expand the analysis to the whole dataset.

In [19]:
results = {}

for outcome, df in datasets.items():
    records = []

    for country, country_data in df.groupby("country", sort=True):
        records.extend(analyse_country(country_data))

    results[outcome] = pd.DataFrame(records)

    print(outcome)
    print(
        results[outcome].drop(columns="dropped_source_rows").round(3).to_string(index=False)
    )

profit
    country     N                  target  estimate     se  dropped  percent  predicted_target  refit_estimate  refit_se  achieved
     Bosnia  1195             Sign change    37.534 19.780       14    1.172            -1.339          -2.226    15.628      True
     Bosnia  1195     Significance change    37.534 19.780        1    0.084             6.656          43.732    18.889      True
     Bosnia  1195 Significant sign change    37.534 19.780       40    3.347            -0.060         -34.929    14.323      True
   Ethiopia  3113             Sign change     7.289  7.893        1    0.032            -0.049          -0.053     2.513      True
   Ethiopia  3113     Significance change     7.289  7.893       45    1.446             0.007          15.356     7.763      True
   Ethiopia  3113 Significant sign change     7.289  7.893       66    2.120            -0.013          -8.755     1.852      True
      India  6863             Sign change    16.722 11.830        6    0.087